# Q7 — Async Data Pipeline

This notebook demonstrates asynchronous programming applied to the data pipeline.

We use `aiosqlite` to write to the database asynchronously and `asyncio.gather()` to run five concurrent reads.

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

import asyncio
import time

from app.async_pipeline import (
    async_insert_indicators,
    async_read_commodity,
    async_read_concurrent,
    run_async_pipeline,
)
from app.pipeline import load_csv, filter_data
from app.calculations import compute_all_indicators

## 1. Async Write

`async_insert_indicators` writes to SQLite using `aiosqlite`, which runs the database operation in a thread pool executor. This means the event loop is not blocked while waiting for the write to complete and so other routines can run concurrently.

The benefit mostly becomes clear when multiple operations are running simultaneously (e.g. writing to multiple tables or databases at the same time).

In [2]:
df_raw = load_csv()
df_filtered = filter_data(df_raw, ["copper", "zinc", "crude_oil"], [2020, 2021])
df_indicators = compute_all_indicators(df_filtered)

start = time.time()
rows  = await async_insert_indicators(df_indicators)
print(f"Inserted {rows} rows in {time.time() - start:.3f}s")

17:25:18 | INFO | Async insert complete — 1569 records processed


Inserted 1569 rows in 0.010s


## 2. Five Concurrent Reads with asyncio.gather()

`asyncio.gather()` dispatches all coroutines at the same time and waits for all of them to complete. This is more efficient than running them sequentially when the operations are independent; each read does not depend on the result of another.

Here we run five database reads concurrently and measure the time compared to running them sequentially.

In [3]:
# Sequential reads
start = time.time()
for commodity in ["copper", "zinc", "crude_oil", "copper", "zinc"]:
    rows = await async_read_commodity(commodity)
sequential_time = time.time() - start
print(f"Sequential: {sequential_time:.3f}s")

# Concurrent reads
start = time.time()
results = await async_read_concurrent()
concurrent_time = time.time() - start
print(f"Concurrent: {concurrent_time:.3f}s")
print(f"Results: {results}")

Sequential: 0.008s
Concurrent: 0.020s
Results: {'read_1_copper': 523, 'read_2_zinc': 523, 'read_3_crude_oil': 523, 'read_4_copper': 523, 'read_5_zinc': 523}


**Note — the concurrent reads were actually slower here, and that is expected.**

Each SQLite read on a local file only takes about 1–2ms. By the time Python dispatches five coroutines, spins up threads for each one (which is what `aiosqlite` does under the hood), and the event loop switches between them, the overhead costs more than just reading five times in a row.

Async concurrency saves time when the individual operations are slow and external — for example, a remote database where each query takes 100ms. Overlapping five of those saves roughly 400ms. But when each operation already finishes in 1ms, there is nothing meaningful to overlap, and the coordination cost dominates.

So the result here is not a bug or a failure of async — it is the correct outcome for this specific situation: small, local, fast reads.

## 3. Run the Full Async Pipeline

In [4]:
await run_async_pipeline()

17:25:20 | INFO | Async pipeline started
17:25:20 | INFO | Async insert complete — 1569 records processed
17:25:20 | INFO | Inserted 1569 rows asynchronously
17:25:20 | INFO | Concurrent reads complete: {'read_1_copper': 523, 'read_2_zinc': 523, 'read_3_crude_oil': 523, 'read_4_copper': 523, 'read_5_zinc': 523}
17:25:20 | INFO | Async pipeline complete


## 4. When Async is Useful and When It Is Not

### When async helps

Async is designed for tasks where the program spends time waiting for something external: a database query, a network request, a file read. While one coroutine is waiting, the event loop can run another. This makes async efficient for mltiple concurrent database queries, fetching data from multiple APIs simultaneously, handling many concurrent HTTP requests in a web server.


### When async does not help

Async provides no benefit  for tasks where the program is actively computing rather than waiting. In our pipeline:

- Loading the CSV with pandas — the computer is actively reading and processing the file
- Computing moving averages, RSI, MACD — the computer is actively doing maths
- Filtering and transforming the data — the computer is actively reorganising the data

Async only helps when the program is waiting for something external, like a network request or a slow API response. While it waits, it can go do other things. But here there is nothing to wait for, so async adds no benefit.

### SQLite limitations

SQLite has some limitations when multiple things try to use it at the same time.

The main issue is that only one thing can write to the database at a time. If two things try to write at the same moment, the second one has to wait for the first one to finish.

Reading is less of a problem as multiple things can read from the database at the same time without blocking each other. For a small project like this, SQLite is fine. But in a real production system where hundreds of users might be writing data at the same time, you would switch to something like PostgreSQL, which handles multiple writers at once much more efficiently.